# Enhanced XAI-IDS Training Pipeline


In [ ]:
!pip -q install lightgbm shap scikit-learn pandas numpy matplotlib seaborn joblib xgboost catboost lime optuna
from google.colab import drive
drive.mount('/content/drive')

## 1. Imports & Configuration

In [ ]:
import os
import glob
import json
import time
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)
from sklearn.metrics.pairwise import cosine_similarity

import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier
import shap
import joblib

# Optional: LIME for cross-validation
try:
    import lime
    import lime.lime_tabular
    LIME_AVAILABLE = True
except ImportError:
    LIME_AVAILABLE = False
    print("LIME not installed. LIME cross-validation will be skipped.")



RANDOM_STATE = 42
RANDOM_SEEDS = [42, 123, 456]          # Multi-seed evaluation
TOP_K_SIGNATURE_FEATURES = 15
MIN_ROWS_PER_CLASS_FOR_SIGNATURE = 10
MAX_SHAP_ROWS = 3000
FEATURE_SELECTION_TOP_N = 20           # SHAP-guided feature selection

# Split settings
MIN_CLASS_COUNT = 30
MIN_LABELS_PER_SPLIT = 4
SEEN_HOST_RATIO = 0.7
MAX_SPLIT_TRIES = 30

# Safe decision constraints
MIN_ATTACK_DETECTION_RATE = 0.95       # Minimum acceptable attack detection

# Paths
DATA_FOLDER = "/content/drive/MyDrive/BCCC-CIC-IDS-2017"
DRIVE_RESULTS_PATH = "/content/drive/MyDrive/IDS_XAI_Project/results_V7"

print("=== Configuration ===")
for k, v in {
    "RANDOM_SEEDS": RANDOM_SEEDS,
    "FEATURE_SELECTION_TOP_N": FEATURE_SELECTION_TOP_N,
    "MIN_ATTACK_DETECTION_RATE": MIN_ATTACK_DETECTION_RATE,
    "MAX_SHAP_ROWS": MAX_SHAP_ROWS,
    "MIN_CLASS_COUNT": MIN_CLASS_COUNT,
    "SEEN_HOST_RATIO": SEEN_HOST_RATIO,
}.items():
    print(f"  {k} = {v}")

In [ ]:
os.makedirs(DRIVE_RESULTS_PATH, exist_ok=True)
print("Results will be saved to:", DRIVE_RESULTS_PATH)

## 2. Data Loading & Cleaning

In [ ]:
csv_files = sorted(glob.glob(os.path.join(DATA_FOLDER, "*.csv")))
assert len(csv_files) > 0, f"No CSV files found in {DATA_FOLDER}"

print("Files found:")
for f in csv_files:
    print(" -", os.path.basename(f))

dfs = []
for f in csv_files:
    df = pd.read_csv(f, low_memory=False)
    df["source_file"] = os.path.basename(f)
    dfs.append(df)
    print(f"Loaded {os.path.basename(f)} -> {df.shape}")

data = pd.concat(dfs, ignore_index=True)
data.columns = [c.strip() for c in data.columns]

print("\nCombined shape:", data.shape)
print("\nColumns:")
print(data.columns.tolist())

In [ ]:
required_cols = ["flow_id", "timestamp", "src_ip", "dst_ip", "label"]
for c in required_cols:
    assert c in data.columns, f"Missing required column: {c}"

data["timestamp"] = pd.to_datetime(data["timestamp"], errors="coerce")
data["label"] = data["label"].astype(str).str.strip()

data = data.dropna(subset=["timestamp", "src_ip", "label"]).reset_index(drop=True)
data = data.drop_duplicates().reset_index(drop=True)

data["Date"] = data["timestamp"].dt.date
data["HostID"] = data["src_ip"].astype(str)

print("Shape after cleaning:", data.shape)
print("Unique dates:", data["Date"].nunique())
print("Unique hosts:", data["HostID"].nunique())
print("Unique labels:", data["label"].nunique())
print("\nTop label counts:")
print(data["label"].value_counts().head(20))

## 3. Train / Validation / Test Splitting

In [ ]:

# 1. Remove tiny classes

label_counts = data["label"].value_counts()
keep_labels = label_counts[label_counts >= MIN_CLASS_COUNT].index
dropped_labels = label_counts[label_counts < MIN_CLASS_COUNT]

if len(dropped_labels) > 0:
    print("Dropping tiny classes:")
    print(dropped_labels)

data_split = data[data["label"].isin(keep_labels)].copy()

print("\nRemaining labels:")
print(data_split["label"].value_counts())



# 2. Split EACH scenario file by time (60/20/20)

train_parts, val_parts, test_parts = [], [], []

for fname, df_file in data_split.groupby("source_file"):
    df_file = df_file.sort_values("timestamp")
    n = len(df_file)
    train_end = int(0.6 * n)
    val_end = int(0.8 * n)

    train_parts.append(df_file.iloc[:train_end])
    val_parts.append(df_file.iloc[train_end:val_end])
    test_parts.append(df_file.iloc[val_end:])

train_df = pd.concat(train_parts).copy()
val_base_df = pd.concat(val_parts).copy()
test_base_df = pd.concat(test_parts).copy()

print("\nAfter scenario time split:")
print("Train rows:", len(train_df))
print("Validation rows:", len(val_base_df))
print("Test rows:", len(test_base_df))



# 3. Host split helper

def split_hosts(df, seen_ratio=0.7, random_state=42):
    hosts = np.array(sorted(df["HostID"].unique()))
    rng = np.random.default_rng(random_state)
    rng.shuffle(hosts)

    n_seen = max(1, int(len(hosts) * seen_ratio))
    seen_hosts = set(hosts[:n_seen])

    seen_df = df[df["HostID"].isin(seen_hosts)].copy()
    unseen_df = df[~df["HostID"].isin(seen_hosts)].copy()

    return seen_df, unseen_df


# 4. Evaluate multiple host split ratios

def print_split_info(name, df):
    print(f"\n{name}")
    print("Rows:", len(df))
    print("Hosts:", df["HostID"].nunique())
    print("Labels:", df["label"].nunique())
    print(df["label"].value_counts())


def evaluate_host_split_ratios(val_df_in, test_df_in, ratios=(0.7, 0.6, 0.5, 0.4), random_state=42):
    rows = []

    for ratio in ratios:
        val_seen_df, host_unseen_df = split_hosts(val_df_in, seen_ratio=ratio, random_state=random_state)
        time_seen_df, hard_unseen_df = split_hosts(test_df_in, seen_ratio=ratio, random_state=random_state)

        rows.append({
            "seen_ratio": ratio,
            "unseen_ratio": round(1 - ratio, 2),
            "val_rows": len(val_seen_df),
            "val_hosts": val_seen_df["HostID"].nunique(),
            "val_labels": val_seen_df["label"].nunique(),
            "host_rows": len(host_unseen_df),
            "host_hosts": host_unseen_df["HostID"].nunique(),
            "host_labels": host_unseen_df["label"].nunique(),
            "time_rows": len(time_seen_df),
            "time_hosts": time_seen_df["HostID"].nunique(),
            "time_labels": time_seen_df["label"].nunique(),
            "hard_rows": len(hard_unseen_df),
            "hard_hosts": hard_unseen_df["HostID"].nunique(),
            "hard_labels": hard_unseen_df["label"].nunique(),
        })

    return pd.DataFrame(rows)


ratio_results = evaluate_host_split_ratios(
    val_base_df,
    test_base_df,
    ratios=(0.7, 0.6, 0.5, 0.4),
    random_state=RANDOM_STATE
)

print("\nHost split ratio experiment summary:")
display(ratio_results)



# 5. Choose the best host split ratio
#    Priority:
#    1) keep full label coverage in VALIDATION
#    2) keep full label coverage in TIME-TEST
#    3) maximize labels in HOST-TEST
#    4) maximize labels in HARD-TEST
#    5) keep more validation rows

full_label_count = data_split["label"].nunique()

ratio_results["val_label_loss"] = full_label_count - ratio_results["val_labels"]
ratio_results["time_label_loss"] = full_label_count - ratio_results["time_labels"]

ratio_results_sorted = ratio_results.sort_values(
    by=[
        "val_label_loss",   # smaller is better
        "time_label_loss",  # smaller is better
        "host_labels",      # larger is better
        "hard_labels",      # larger is better
        "val_rows",         # larger is better
        "time_rows"         # larger is better
    ],
    ascending=[True, True, False, False, False, False]
).reset_index(drop=True)

best_seen_ratio = ratio_results_sorted.loc[0, "seen_ratio"]
best_unseen_ratio = ratio_results_sorted.loc[0, "unseen_ratio"]

print("\nSelected host split ratio:")
print(f"Seen/Unseen = {best_seen_ratio:.1f}/{best_unseen_ratio:.1f}")

print("\nSorted ratio comparison:")
display(ratio_results_sorted)

In [ ]:
# Apply selected host split ratio to validation and test
val_df, host_test_df = split_hosts(val_base_df, seen_ratio=best_seen_ratio, random_state=RANDOM_STATE)
time_test_df, hard_test_df = split_hosts(test_base_df, seen_ratio=best_seen_ratio, random_state=RANDOM_STATE)

print("Final dataset shapes:")
print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Host-Test:", host_test_df.shape)
print("Time-Test:", time_test_df.shape)
print("Hard-Test:", hard_test_df.shape)

## 4. Feature Engineering & Robust Label Encoding

In [ ]:
leakage_cols = [
    "flow_id", "timestamp", "Date", "src_ip", "dst_ip",
    "HostID", "label", "source_file"
]

numeric_feature_cols = [
    c for c in train_df.columns
    if c not in leakage_cols and pd.api.types.is_numeric_dtype(train_df[c])
]

assert len(numeric_feature_cols) > 0, "No numeric model features found."
print("Number of raw features:", len(numeric_feature_cols))


def clean_X(df, feature_cols):
    X = df[feature_cols].copy()
    X = X.replace([np.inf, -np.inf], np.nan)
    return X


X_train = clean_X(train_df, numeric_feature_cols)
X_val = clean_X(val_df, numeric_feature_cols)
X_host_test = clean_X(host_test_df, numeric_feature_cols)
X_time_test = clean_X(time_test_df, numeric_feature_cols)
X_hard_test = clean_X(hard_test_df, numeric_feature_cols)


# Robust Label Encoding: handle unknown labels gracefully

le = LabelEncoder()
y_train_raw = train_df["label"].astype(str).values
le.fit(y_train_raw)

known_classes = set(le.classes_)


def safe_transform(labels, label_encoder, unknown_label="UNKNOWN"):
    """Transform labels, mapping any unseen labels to a dedicated UNKNOWN class."""
    labels = np.array(labels, dtype=str)
    known = set(label_encoder.classes_)
    unknown_mask = np.array([l not in known for l in labels])

    if unknown_mask.any():
        n_unknown = unknown_mask.sum()
        unknown_names = set(labels[unknown_mask])
        print(f"  WARNING: {n_unknown} samples with unknown labels: {unknown_names}")
        print(f"  These will be mapped to '{unknown_label}'.")

        # Extend encoder with unknown class
        new_classes = np.append(label_encoder.classes_, unknown_label)
        label_encoder.classes_ = new_classes
        labels[unknown_mask] = unknown_label

    return label_encoder.transform(labels)


y_train = le.transform(y_train_raw)
y_val = safe_transform(val_df["label"].astype(str).values, le)
y_host_test = safe_transform(host_test_df["label"].astype(str).values, le)
y_time_test = safe_transform(time_test_df["label"].astype(str).values, le)
y_hard_test = safe_transform(hard_test_df["label"].astype(str).values, le)

print("\nClasses:", le.classes_)

benign_candidates = [i for i, c in enumerate(le.classes_) if "benign" in c.lower()]
benign_idx = benign_candidates[0] if benign_candidates else None
print("\nBenign index:", benign_idx)
if benign_idx is not None:
    print("Benign class:", le.classes_[benign_idx])

## 5. Model Training (LightGBM, XGBoost, CatBoost, Random Forest, MLP)

In [ ]:
n_classes = len(le.classes_)
print(f"Training {n_classes}-class models...\n")
timing = {}


# --- 5a. LightGBM ---
t0 = time.time()
lgb_model = lgb.LGBMClassifier(
    objective="multiclass",
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=64,
    random_state=RANDOM_STATE,
    class_weight="balanced",
    n_jobs=-1,
    verbose=-1
)
lgb_model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    eval_metric="multi_logloss",
    callbacks=[lgb.early_stopping(30), lgb.log_evaluation(50)]
)
timing["LightGBM_train"] = time.time() - t0
print(f"LightGBM trained in {timing['LightGBM_train']:.1f}s\n")


# --- 5b. XGBoost ---
t0 = time.time()
xgb_model = xgb.XGBClassifier(
    objective="multi:softprob",
    n_estimators=300,
    learning_rate=0.05,
    max_depth=8,
    random_state=RANDOM_STATE,
    use_label_encoder=False,
    eval_metric="mlogloss",
    n_jobs=-1,
    verbosity=0
)
# Compute sample weights for class balance
from sklearn.utils.class_weight import compute_sample_weight
sample_weights = compute_sample_weight("balanced", y_train)
xgb_model.fit(
    X_train, y_train,
    sample_weight=sample_weights,
    eval_set=[(X_val, y_val)],
    verbose=False
)
timing["XGBoost_train"] = time.time() - t0
print(f"XGBoost trained in {timing['XGBoost_train']:.1f}s\n")


# --- 5c. CatBoost ---
t0 = time.time()
cb_model = CatBoostClassifier(
    iterations=300,
    learning_rate=0.05,
    depth=8,
    random_seed=RANDOM_STATE,
    auto_class_weights="Balanced",
    eval_metric="MultiClass",
    verbose=50
)
cb_model.fit(
    X_train, y_train,
    eval_set=(X_val, y_val),
    early_stopping_rounds=30
)
timing["CatBoost_train"] = time.time() - t0
print(f"CatBoost trained in {timing['CatBoost_train']:.1f}s\n")


# --- 5d. Random Forest ---
t0 = time.time()
# Impute NaN before RF (it does not handle NaN natively)
from sklearn.impute import SimpleImputer
rf_imputer = SimpleImputer(strategy="median")
X_train_imputed = rf_imputer.fit_transform(X_train)

rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=20,
    random_state=RANDOM_STATE,
    class_weight="balanced",
    n_jobs=-1
)
rf_model.fit(X_train_imputed, y_train)
timing["RandomForest_train"] = time.time() - t0
print(f"Random Forest trained in {timing['RandomForest_train']:.1f}s\n")


# --- 5e. MLP ---
t0 = time.time()
mlp_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("clf", MLPClassifier(
        hidden_layer_sizes=(128, 64),
        activation="relu",
        solver="adam",
        batch_size=256,
        max_iter=50,
        random_state=RANDOM_STATE,
        early_stopping=True,
        validation_fraction=0.1
    ))
])
mlp_model.fit(X_train, y_train)
timing["MLP_train"] = time.time() - t0
print(f"MLP trained in {timing['MLP_train']:.1f}s\n")


print("\n=== Training Times ===")
for name, t in timing.items():
    print(f"  {name}: {t:.1f}s")

## 6. Probability Calibration

In [ ]:
# Calibrate the primary model (LightGBM) using isotonic regression on the val set.
# This improves reliability of probability-based thresholds.

print("Calibrating LightGBM probabilities with isotonic regression...")
lgb_calibrated = CalibratedClassifierCV(
    lgb_model,
    method="isotonic",
    cv="prefit"  # use already-trained model
)
lgb_calibrated.fit(X_val, y_val)
print("Calibration complete.\n")

# Quick calibration check
raw_proba = lgb_model.predict_proba(X_val)
cal_proba = lgb_calibrated.predict_proba(X_val)

print("Raw LightGBM max confidence (mean):", raw_proba.max(axis=1).mean().round(4))
print("Calibrated LightGBM max confidence (mean):", cal_proba.max(axis=1).mean().round(4))

## 7. SHAP-Based Feature Selection

In [ ]:
# Train a preliminary SHAP explainer and select top features to reduce noise
# in signature vectors and improve cosine similarity reliability.

print(f"Computing SHAP importance on {min(MAX_SHAP_ROWS, len(X_train))} training samples...")


def sample_df(X, y, max_rows=3000, random_state=42):
    """Subsample dataframe for SHAP computation."""
    if len(X) <= max_rows:
        return X.copy(), np.array(y)
    rng = np.random.default_rng(random_state)
    idx = rng.choice(len(X), size=max_rows, replace=False)
    return X.iloc[idx].copy(), np.array(y)[idx]


X_sel_shap, y_sel_shap = sample_df(X_train, y_train, MAX_SHAP_ROWS, RANDOM_STATE)
explainer_full = shap.TreeExplainer(lgb_model)
shap_raw_full = explainer_full.shap_values(X_sel_shap)

# Aggregate absolute SHAP importance across all classes
if isinstance(shap_raw_full, list):
    # Multi-output: list of arrays per class
    abs_shap = np.mean([np.abs(sv) for sv in shap_raw_full], axis=0)
else:
    abs_shap = np.abs(shap_raw_full)
    if abs_shap.ndim == 3:
        abs_shap = abs_shap.mean(axis=2)  # average over classes

global_importance = abs_shap.mean(axis=0)
feature_importance_df = pd.DataFrame({
    "feature": numeric_feature_cols,
    "shap_importance": global_importance
}).sort_values("shap_importance", ascending=False)

# Select top N features
selected_features = feature_importance_df.head(FEATURE_SELECTION_TOP_N)["feature"].tolist()
dropped_features = [f for f in numeric_feature_cols if f not in selected_features]

print(f"\nSelected {len(selected_features)} features (from {len(numeric_feature_cols)}):")
print(feature_importance_df.head(FEATURE_SELECTION_TOP_N).to_string(index=False))
print(f"\nDropped {len(dropped_features)} low-importance features")

# Plot feature importance
plt.figure(figsize=(10, 8))
top_plot = feature_importance_df.head(25)
sns.barplot(data=top_plot, y="feature", x="shap_importance", orient="h")
plt.title("Top 25 Features by SHAP Importance")
plt.tight_layout()
plt.show()


# Rebuild feature matrices with selected features only
X_train = X_train[selected_features]
X_val = X_val[selected_features]
X_host_test = X_host_test[selected_features]
X_time_test = X_time_test[selected_features]
X_hard_test = X_hard_test[selected_features]

print(f"\nFeature matrices reduced to {X_train.shape[1]} columns")

## 8. Retrain All Models on Selected Features

In [ ]:
# Retrain on the reduced feature set for final evaluation

print("Retraining all models on selected features...\n")

# LightGBM
lgb_model = lgb.LGBMClassifier(
    objective="multiclass", n_estimators=300, learning_rate=0.05,
    num_leaves=64, random_state=RANDOM_STATE, class_weight="balanced",
    n_jobs=-1, verbose=-1
)
lgb_model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    eval_metric="multi_logloss",
    callbacks=[lgb.early_stopping(30), lgb.log_evaluation(0)]
)

# Calibrate
lgb_calibrated = CalibratedClassifierCV(lgb_model, method="isotonic", cv="prefit")
lgb_calibrated.fit(X_val, y_val)

# XGBoost
sample_weights = compute_sample_weight("balanced", y_train)
xgb_model = xgb.XGBClassifier(
    objective="multi:softprob", n_estimators=300, learning_rate=0.05,
    max_depth=8, random_state=RANDOM_STATE, use_label_encoder=False,
    eval_metric="mlogloss", n_jobs=-1, verbosity=0
)
xgb_model.fit(X_train, y_train, sample_weight=sample_weights,
              eval_set=[(X_val, y_val)], verbose=False)

# CatBoost
cb_model = CatBoostClassifier(
    iterations=300, learning_rate=0.05, depth=8,
    random_seed=RANDOM_STATE, auto_class_weights="Balanced",
    eval_metric="MultiClass", verbose=0
)
cb_model.fit(X_train, y_train, eval_set=(X_val, y_val), early_stopping_rounds=30)

# Random Forest
X_train_imp = rf_imputer.fit_transform(X_train)
rf_model = RandomForestClassifier(
    n_estimators=200, max_depth=20, random_state=RANDOM_STATE,
    class_weight="balanced", n_jobs=-1
)
rf_model.fit(X_train_imp, y_train)

# MLP
mlp_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("clf", MLPClassifier(
        hidden_layer_sizes=(128, 64), activation="relu", solver="adam",
        batch_size=256, max_iter=50, random_state=RANDOM_STATE,
        early_stopping=True, validation_fraction=0.1
    ))
])
mlp_model.fit(X_train, y_train)

print("All models retrained on", X_train.shape[1], "selected features.")

## 9. Evaluation Utilities

In [ ]:
def fix_shape(y):
    y = np.asarray(y)

    # Convert (n, 1) -> (n,)
    if y.ndim == 2 and y.shape[1] == 1:
        y = y.ravel()

    return y


def evaluate_predictions(y_true, y_pred, label_encoder, title=""):
    y_true = fix_shape(y_true)
    y_pred = fix_shape(y_pred)

    print(f"\n===== {title} =====")
    print("Accuracy:", round(accuracy_score(y_true, y_pred), 4))
    print("Macro F1:", round(f1_score(y_true, y_pred, average="macro", zero_division=0), 4))
    print("Weighted F1:", round(f1_score(y_true, y_pred, average="weighted", zero_division=0), 4))

    present_labels = sorted(np.unique(np.concatenate([y_true, y_pred])))
    present_target_names = [label_encoder.classes_[i] for i in present_labels]

    print("\nClassification Report:")
    print(classification_report(
        y_true, y_pred,
        labels=present_labels,
        target_names=present_target_names,
        zero_division=0
    ))


def collect_metrics(y_true, y_pred, model_name, dataset_name):
    y_true = fix_shape(y_true)
    y_pred = fix_shape(y_pred)

    present_labels = np.unique(np.concatenate([y_true, y_pred]))

    return {
        "Model": model_name,
        "Dataset": dataset_name,
        "Accuracy": accuracy_score(y_true, y_pred),
        "Macro_Precision": precision_score(y_true, y_pred, labels=present_labels, average="macro", zero_division=0),
        "Macro_Recall": recall_score(y_true, y_pred, labels=present_labels, average="macro", zero_division=0),
        "Macro_F1": f1_score(y_true, y_pred, labels=present_labels, average="macro", zero_division=0),
        "Weighted_Precision": precision_score(y_true, y_pred, labels=present_labels, average="weighted", zero_division=0),
        "Weighted_Recall": recall_score(y_true, y_pred, labels=present_labels, average="weighted", zero_division=0),
        "Weighted_F1": f1_score(y_true, y_pred, labels=present_labels, average="weighted", zero_division=0),
    }

## 10. Model Evaluation (All 5 Models x 4 Datasets)

In [ ]:
# Define model dict for iteration
models = {
    "LightGBM": lgb_model,
    "XGBoost": xgb_model,
    "CatBoost": cb_model,
    "MLP": mlp_model,
}

# RF needs imputed data
def rf_predict(X):
    return rf_model.predict(rf_imputer.transform(X))

datasets = {
    "Validation": (X_val, y_val),
    "Host-Test": (X_host_test, y_host_test),
    "Time-Test": (X_time_test, y_time_test),
    "Hard-Test": (X_hard_test, y_hard_test),
}

metrics = []
inference_times = []

for model_name, model in models.items():
    for ds_name, (X_ds, y_ds) in datasets.items():
        t0 = time.time()
        y_pred = fix_shape(model.predict(X_ds))
        t_infer = time.time() - t0

        metrics.append(collect_metrics(y_ds, y_pred, model_name, ds_name))
        inference_times.append({
            "Model": model_name,
            "Dataset": ds_name,
            "Samples": len(X_ds),
            "Time_s": round(t_infer, 4),
            "Throughput_sps": round(len(X_ds) / max(t_infer, 1e-6), 0)
        })

        evaluate_predictions(y_ds, y_pred, le, f"{model_name} - {ds_name}")

# Random Forest (special handling for NaN)
for ds_name, (X_ds, y_ds) in datasets.items():
    t0 = time.time()
    y_pred = fix_shape(rf_predict(X_ds))
    t_infer = time.time() - t0

    metrics.append(collect_metrics(y_ds, y_pred, "RandomForest", ds_name))
    inference_times.append({
        "Model": "RandomForest",
        "Dataset": ds_name,
        "Samples": len(X_ds),
        "Time_s": round(t_infer, 4),
        "Throughput_sps": round(len(X_ds) / max(t_infer, 1e-6), 0)
    })

    evaluate_predictions(y_ds, y_pred, le, f"RandomForest - {ds_name}")

In [ ]:
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd

def plot_confusion(y_true, y_pred, label_encoder, title):
    cm = confusion_matrix(y_true, y_pred)
    labels = label_encoder.classes_

    plt.figure(figsize=(10, 8))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=labels,
        yticklabels=labels
    )
    plt.title(title)
    plt.xlabel("Predicted Label")
    plt.ylabel("True Label")
    plt.xticks(rotation=90)
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.show()


# Macro F1 comparison across all models and datasets
metrics_df = pd.DataFrame(metrics)

plt.figure(figsize=(12, 6))
sns.barplot(data=metrics_df, x="Dataset", y="Macro_F1", hue="Model")
plt.title("Attack Detection Performance (Macro F1) - All Models")
plt.ylabel("Macro F1 Score")
plt.xlabel("Dataset")
plt.legend(title="Model")
plt.tight_layout()
plt.show()


# Confusion matrices for best model on test sets
# (Host-Test optional since it only has 2 classes)
for ds_name in ["Time-Test", "Hard-Test"]:
    X_ds, y_ds = datasets[ds_name]
    y_pred = lgb_model.predict(X_ds)
    plot_confusion(y_ds, y_pred, le, f"LightGBM Confusion Matrix - {ds_name}")

## 11. SHAP Signatures & Explanation Stability

In [ ]:
def convert_shap_to_predicted_class_vectors(shap_values, pred_class_idx):
    """Extract SHAP vector for the predicted class of each sample."""
    if isinstance(shap_values, list):
        return np.array([shap_values[c][i] for i, c in enumerate(pred_class_idx)])

    shap_values = np.array(shap_values)
    if shap_values.ndim == 3:
        return np.array([shap_values[i, :, c] for i, c in enumerate(pred_class_idx)])
    if shap_values.ndim == 2:
        return shap_values
    raise ValueError(f"Unexpected SHAP shape: {shap_values.shape}")


def compute_shap_vectors(X, explainer, model):
    """Compute SHAP vectors for predicted class."""
    shap_raw = explainer.shap_values(X)
    pred_idx = np.argmax(model.predict_proba(X), axis=1)
    return convert_shap_to_predicted_class_vectors(shap_raw, pred_idx), pred_idx


# Create SHAP explainer on selected features
explainer = shap.TreeExplainer(lgb_model)

# Sample and compute SHAP for each split
shap_data = {}
for name, (X_ds, y_ds) in [("Train", (X_train, y_train))] + list(datasets.items()):
    X_s, y_s = sample_df(X_ds, y_ds, MAX_SHAP_ROWS, RANDOM_STATE)
    vecs, pred_idx = compute_shap_vectors(X_s, explainer, lgb_model)
    shap_data[name] = {"X": X_s, "y": y_s, "vecs": vecs, "pred_idx": pred_idx}
    print(f"{name}: SHAP shape = {vecs.shape}")

# Summary plot
plt.figure()
train_shap_raw = explainer.shap_values(shap_data["Train"]["X"])
shap.summary_plot(train_shap_raw, shap_data["Train"]["X"], show=False, max_display=FEATURE_SELECTION_TOP_N)
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.preprocessing import normalize
def build_class_signatures(shap_vectors, y_true, label_encoder, min_rows=10):
    """Build mean SHAP signature per class."""
    signatures = {}
    counts = {}
    for class_idx in np.unique(y_true):
        mask = (y_true == class_idx)
        if mask.sum() >= min_rows:
            class_name = label_encoder.inverse_transform([class_idx])[0]
            sig = np.median(shap_vectors[mask], axis=0)
            sig = normalize(sig.reshape(1, -1))[0]
            signatures[class_name] = sig
            #signatures[class_name] = np.median(shap_vectors[mask], axis=0)
            counts[class_name] = int(mask.sum())
    return signatures, counts


def top_k_signature_features(signature_vector, feature_names, k=8):
    idx = np.argsort(np.abs(signature_vector))[-k:][::-1]
    return [(feature_names[i], float(signature_vector[i])) for i in idx]


# Build signatures for each split
signatures = {}
for name in shap_data:
    sigs, counts = build_class_signatures(
        shap_data[name]["vecs"], shap_data[name]["y"],
        le, MIN_ROWS_PER_CLASS_FOR_SIGNATURE
    )
    signatures[name] = sigs
    print(f"\n{name} signatures: {list(sigs.keys())}")

# Print top features for training signatures
print("\nTop features per training signature:")
for attack_name, sig in list(signatures["Train"].items())[:10]:
    print(f"\n  {attack_name}:")
    for feat, val in top_k_signature_features(sig, selected_features, k=TOP_K_SIGNATURE_FEATURES):
        print(f"    {feat:35s} {val:+.5f}")

In [ ]:
def signature_similarity_table(train_sigs, test_sigs):
    rows = []
    for name, train_sig in train_sigs.items():
        if name not in test_sigs:
            continue
        same = cosine_similarity([train_sig], [test_sigs[name]])[0, 0]
        other_sims = [
            cosine_similarity([train_sig], [other_sig])[0, 0]
            for other_name, other_sig in test_sigs.items()
            if other_name != name
        ]
        mean_other = np.mean(other_sims) if other_sims else np.nan
        rows.append({
            "Attack": name,
            "SameClassSimilarity": same,
            "OtherClassMeanSimilarity": mean_other,
            "Gap": same - mean_other if not np.isnan(mean_other) else np.nan
        })
    if not rows:
        return pd.DataFrame(columns=["Attack", "SameClassSimilarity", "OtherClassMeanSimilarity", "Gap"])
    return pd.DataFrame(rows).sort_values("Gap", ascending=False)


for test_name in ["Host-Test", "Time-Test", "Hard-Test"]:
    sim_table = signature_similarity_table(signatures["Train"], signatures.get(test_name, {}))
    print(f"\nTrain vs {test_name}:")
    display(sim_table.round(4))
    sim_table.to_csv(os.path.join(DRIVE_RESULTS_PATH, f"{test_name.lower()}_signature_stability.csv"), index=False)


# Signature similarity matrix (heatmap)
train_sigs = signatures["Train"]
attack_names = list(train_sigs.keys())
if len(attack_names) > 1:
    matrix = np.array([
        [cosine_similarity([train_sigs[a]], [train_sigs[b]])[0, 0]
         for b in attack_names]
        for a in attack_names
    ])
    plt.figure(figsize=(10, 8))
    sns.heatmap(pd.DataFrame(matrix, index=attack_names, columns=attack_names),
                cmap="coolwarm", center=0.5)
    plt.title("Attack Signature Similarity Matrix (Train)")
    plt.tight_layout()
    plt.show()

## 12. Safe Decision Policy (Enhanced)

In [ ]:
assert benign_idx is not None, "A benign class is required for the safe-decision layer."


# Store train signatures by class index
train_sig_by_class_idx = {}
for label_name, sig in signatures["Train"].items():
    class_idx = le.transform([label_name])[0]
    train_sig_by_class_idx[class_idx] = sig


def compute_pred_similarities_vectorized(shap_vecs, pred_idx, train_sig_by_class_idx):
    """Vectorized cosine similarity between each sample's SHAP vector
    and the training signature of its predicted class.

    ~10-50x faster than row-by-row computation.
    """
    n = len(shap_vecs)
    sims = np.full(n, -1.0)

    for class_idx, sig in train_sig_by_class_idx.items():
        mask = (pred_idx == class_idx)
        if mask.any():
            class_vecs = shap_vecs[mask]
            # Vectorized cosine similarity: (n_samples, n_features) vs (1, n_features)
            sims[mask] = cosine_similarity(class_vecs, sig.reshape(1, -1)).ravel()

    return sims


def evaluate_safe_decision_policy(proba, pred_idx, true_idx, benign_idx,
                                  pred_signature_similarity,
                                  prob_threshold, sim_threshold):
    """Three-tier safe decision: ALLOW / BLOCK / ESCALATE."""
    attack_conf = 1 - proba[:, benign_idx]

    decisions = np.full(len(true_idx), "ESCALATE", dtype=object)

    # ALLOW: predicted benign with low attack confidence
    allow_mask = (pred_idx == benign_idx) & (attack_conf < prob_threshold)
    decisions[allow_mask] = "ALLOW"

    # BLOCK: predicted attack with high confidence AND high signature similarity
    block_mask = (
        (pred_idx != benign_idx) &
        (attack_conf >= prob_threshold) &
        (pred_signature_similarity >= sim_threshold)
    )
    decisions[block_mask] = "BLOCK"

    # Everything else remains ESCALATE

    true_attack = (true_idx != benign_idx)
    true_benign = ~true_attack

    n_true_attack = max(np.sum(true_attack), 1)
    n_true_benign = max(np.sum(true_benign), 1)

    missed_attacks = int(np.sum((decisions == "ALLOW") & true_attack))
    attack_not_missed_rate = np.sum((decisions != "ALLOW") & true_attack) / n_true_attack
    false_block_rate = np.sum((decisions == "BLOCK") & true_benign) / n_true_benign
    escalation_rate = np.mean(decisions == "ESCALATE")

    # Per-class metrics
    per_class = {}
    for c in np.unique(true_idx):
        c_mask = (true_idx == c)
        c_name = le.inverse_transform([c])[0]
        if c == benign_idx:
            per_class[c_name] = {
                "n": int(c_mask.sum()),
                "correctly_allowed": int(np.sum((decisions == "ALLOW") & c_mask)),
                "false_blocked": int(np.sum((decisions == "BLOCK") & c_mask)),
                "escalated": int(np.sum((decisions == "ESCALATE") & c_mask)),
            }
        else:
            per_class[c_name] = {
                "n": int(c_mask.sum()),
                "missed": int(np.sum((decisions == "ALLOW") & c_mask)),
                "correctly_blocked": int(np.sum((decisions == "BLOCK") & c_mask)),
                "escalated": int(np.sum((decisions == "ESCALATE") & c_mask)),
            }

    return {
        "missed_attacks": missed_attacks,
        "attack_not_missed_rate": attack_not_missed_rate,
        "false_block_rate": false_block_rate,
        "escalation_rate": escalation_rate,
        "per_class": per_class,
    }


print("Safe decision functions defined (vectorized).")

## 13. Automatic Threshold Search (Constrained)

In [ ]:
# Prepare validation SHAP + confidence for threshold search

X_val_shap, y_val_shap = sample_df(X_val, y_val, MAX_SHAP_ROWS, RANDOM_STATE)
val_proba = lgb_calibrated.predict_proba(X_val_shap)  # Use CALIBRATED probabilities
val_pred_idx = np.argmax(val_proba, axis=1)
val_attack_conf = 1 - val_proba[:, benign_idx]

val_shap_raw = explainer.shap_values(X_val_shap)
val_shap_vecs = convert_shap_to_predicted_class_vectors(val_shap_raw, val_pred_idx)

# Vectorized similarity
val_sim = compute_pred_similarities_vectorized(val_shap_vecs, val_pred_idx, train_sig_by_class_idx)

correct_attack_mask = (val_pred_idx == y_val_shap) & (y_val_shap != benign_idx)


# Grid search with MINIMUM DETECTION RATE CONSTRAINT

prob_quantiles = [0.30, 0.40, 0.50, 0.60, 0.70, 0.80]
sim_quantiles = [0.05, 0.10, 0.15, 0.20, 0.25, 0.30]

search_rows = []

for pq in prob_quantiles:
    prob_threshold_candidate = np.quantile(val_attack_conf, pq)

    for sq in sim_quantiles:
        if np.sum(correct_attack_mask) > 10:
            sim_threshold_candidate = np.quantile(val_sim[correct_attack_mask], sq)
        else:
            sim_threshold_candidate = np.quantile(val_sim, sq)

        result = evaluate_safe_decision_policy(
            proba=val_proba,
            pred_idx=val_pred_idx,
            true_idx=y_val_shap,
            benign_idx=benign_idx,
            pred_signature_similarity=val_sim,
            prob_threshold=prob_threshold_candidate,
            sim_threshold=sim_threshold_candidate
        )

        search_rows.append({
            "prob_quantile": pq,
            "sim_quantile": sq,
            "prob_threshold": prob_threshold_candidate,
            "sim_threshold": sim_threshold_candidate,
            "missed_attacks": result["missed_attacks"],
            "attack_not_missed_rate": result["attack_not_missed_rate"],
            "false_block_rate": result["false_block_rate"],
            "escalation_rate": result["escalation_rate"],
        })

threshold_search_df = pd.DataFrame(search_rows)

# *** CRITICAL FIX: Enforce minimum attack detection rate ***
valid_df = threshold_search_df[
    threshold_search_df["attack_not_missed_rate"] >= MIN_ATTACK_DETECTION_RATE
].copy()

if len(valid_df) == 0:
    print(f"WARNING: No threshold pair meets MIN_ATTACK_DETECTION_RATE={MIN_ATTACK_DETECTION_RATE}")
    print("Relaxing to best available attack_not_missed_rate...")
    valid_df = threshold_search_df.copy()

# Sort: lowest false_block_rate, then lowest escalation_rate
valid_df = valid_df.sort_values(
    by=["false_block_rate", "escalation_rate", "missed_attacks"],
    ascending=[True, True, True]
).reset_index(drop=True)

best_row = valid_df.iloc[0]
prob_threshold = best_row["prob_threshold"]
sim_threshold = best_row["sim_threshold"]

print("Best thresholds (with detection rate >= {:.0%}):".format(MIN_ATTACK_DETECTION_RATE))
print(f"  prob_threshold = {prob_threshold:.6f}  (quantile {best_row['prob_quantile']})")
print(f"  sim_threshold  = {sim_threshold:.6f}  (quantile {best_row['sim_quantile']})")
print(f"  attack_not_missed_rate = {best_row['attack_not_missed_rate']:.4f}")
print(f"  false_block_rate       = {best_row['false_block_rate']:.4f}")
print(f"  escalation_rate        = {best_row['escalation_rate']:.4f}")

print("\nFull search results (filtered):")
display(valid_df.round(4))

## 14. Test Safe Decision Policy

In [ ]:
def test_safe_policy(X_test, y_test, name, prob_threshold, sim_threshold):
    """Evaluate safe decision policy on a test set with timing."""
    X_test_small, y_test_small = sample_df(X_test, y_test, MAX_SHAP_ROWS, RANDOM_STATE)

    # Prediction timing
    t0 = time.time()
    proba = lgb_calibrated.predict_proba(X_test_small)
    t_pred = time.time() - t0

    pred_idx = np.argmax(proba, axis=1)

    # SHAP timing
    t0 = time.time()
    shap_raw = explainer.shap_values(X_test_small)
    shap_vecs = convert_shap_to_predicted_class_vectors(shap_raw, pred_idx)
    t_shap = time.time() - t0

    # Similarity timing
    t0 = time.time()
    pred_sim = compute_pred_similarities_vectorized(shap_vecs, pred_idx, train_sig_by_class_idx)
    t_sim = time.time() - t0

    # Baseline (confidence only, no SHAP)
    attack_conf = 1 - proba[:, benign_idx]
    baseline_pred_attack = attack_conf >= prob_threshold
    true_attack = y_test_small != benign_idx
    true_benign = ~true_attack

    baseline_missed = int(np.sum((~baseline_pred_attack) & true_attack))
    baseline_not_missed_rate = np.sum(baseline_pred_attack & true_attack) / max(np.sum(true_attack), 1)
    baseline_false_block_rate = np.sum(baseline_pred_attack & true_benign) / max(np.sum(true_benign), 1)

    # Explain-aware
    safe = evaluate_safe_decision_policy(
        proba=proba, pred_idx=pred_idx, true_idx=y_test_small,
        benign_idx=benign_idx, pred_signature_similarity=pred_sim,
        prob_threshold=prob_threshold, sim_threshold=sim_threshold
    )

    # Print per-class breakdown
    print(f"\n--- {name}: Per-Class Safe Decision Breakdown ---")
    for cls_name, cls_info in safe["per_class"].items():
        print(f"  {cls_name}: {cls_info}")

    total_time = t_pred + t_shap + t_sim
    n = len(X_test_small)

    return pd.DataFrame([{
        "Dataset": name,
        "prob_threshold": prob_threshold,
        "sim_threshold": sim_threshold,
        "Baseline_missed_attacks": baseline_missed,
        "Baseline_attack_not_missed_rate": baseline_not_missed_rate,
        "Baseline_false_block_rate": baseline_false_block_rate,
        "ExplainAware_missed_attacks": safe["missed_attacks"],
        "ExplainAware_attack_not_missed_rate": safe["attack_not_missed_rate"],
        "ExplainAware_false_block_rate": safe["false_block_rate"],
        "ExplainAware_escalation_rate": safe["escalation_rate"],
        "Time_prediction_s": round(t_pred, 3),
        "Time_SHAP_s": round(t_shap, 3),
        "Time_similarity_s": round(t_sim, 3),
        "Time_total_s": round(total_time, 3),
        "Throughput_decisions_per_s": round(n / max(total_time, 1e-6), 0),
    }])


# Apply to all test sets
safe_results = pd.concat([
    test_safe_policy(X_host_test, y_host_test, "Host-Test", prob_threshold, sim_threshold),
    test_safe_policy(X_time_test, y_time_test, "Time-Test", prob_threshold, sim_threshold),
    test_safe_policy(X_hard_test, y_hard_test, "Hard-Test", prob_threshold, sim_threshold),
], ignore_index=True)

print("\nSafe Decision Results:")
display(safe_results.round(4))

## 15. LIME Cross-Validation of SHAP Explanations

In [ ]:
if LIME_AVAILABLE:
    print("Running LIME cross-validation on a sample of predictions...\n")

    # Create LIME explainer on training data
    X_train_np = X_train.fillna(X_train.median()).values
    lime_explainer = lime.lime_tabular.LimeTabularExplainer(
        X_train_np,
        feature_names=selected_features,
        class_names=[str(c) for c in le.classes_],
        mode="classification",
        random_state=RANDOM_STATE,
        discretize_continuous=True
    )

    # Sample a small set for LIME (expensive per-instance)
    N_LIME = 50
    X_lime, y_lime = sample_df(X_host_test, y_host_test, N_LIME, RANDOM_STATE)
    lime_pred_idx = np.argmax(lgb_model.predict_proba(X_lime), axis=1)

    # Get SHAP for the same samples
    lime_shap_raw = explainer.shap_values(X_lime)
    lime_shap_vecs = convert_shap_to_predicted_class_vectors(lime_shap_raw, lime_pred_idx)

    agreement_scores = []
    for i in range(len(X_lime)):
        # SHAP top features
        shap_abs = np.abs(lime_shap_vecs[i])
        shap_top5 = set(np.argsort(shap_abs)[-5:])

        # LIME top features
        row = X_lime.iloc[i].fillna(0).values
        lime_exp = lime_explainer.explain_instance(
            row, lgb_model.predict_proba,
            num_features=5, labels=[int(lime_pred_idx[i])]
        )
        lime_feats = lime_exp.as_map().get(int(lime_pred_idx[i]), [])
        lime_top5 = set([f[0] for f in lime_feats[:5]])

        # Jaccard overlap
        if len(shap_top5 | lime_top5) > 0:
            jaccard = len(shap_top5 & lime_top5) / len(shap_top5 | lime_top5)
        else:
            jaccard = 0.0
        agreement_scores.append(jaccard)

    agreement_scores = np.array(agreement_scores)
    print(f"SHAP-LIME Agreement (Jaccard of top-5 features):")
    print(f"  Mean:   {agreement_scores.mean():.3f}")
    print(f"  Median: {np.median(agreement_scores):.3f}")
    print(f"  Std:    {agreement_scores.std():.3f}")
    print(f"  Min:    {agreement_scores.min():.3f}")
    print(f"  Max:    {agreement_scores.max():.3f}")

    # Low agreement samples -> candidates for ESCALATE
    low_agreement = agreement_scores < 0.2
    print(f"\n  Samples with low SHAP-LIME agreement (<0.2): {low_agreement.sum()} / {len(agreement_scores)}")
    print("  These samples should be flagged for ESCALATION in production.")

    plt.figure(figsize=(8, 4))
    plt.hist(agreement_scores, bins=20, edgecolor="black", alpha=0.7)
    plt.xlabel("SHAP-LIME Top-5 Jaccard Agreement")
    plt.ylabel("Count")
    plt.title("Distribution of SHAP-LIME Explanation Agreement")
    plt.tight_layout()
    plt.show()
else:
    print("LIME not available. Skipping cross-validation.")
    print("Install with: pip install lime")

## 16. Visualization

In [ ]:
# Safe decision comparison plot
safe_plot = safe_results.melt(
    id_vars="Dataset",
    value_vars=[
        "Baseline_attack_not_missed_rate",
        "ExplainAware_attack_not_missed_rate"
    ],
    var_name="Method",
    value_name="AttackDetectionRate"
)
safe_plot["Method"] = safe_plot["Method"].map({
    "Baseline_attack_not_missed_rate": "Baseline (confidence only)",
    "ExplainAware_attack_not_missed_rate": "Explain-Aware (SHAP signatures)"
})

plt.figure(figsize=(10, 6))
sns.barplot(data=safe_plot, x="Dataset", y="AttackDetectionRate", hue="Method")
plt.title("Safe Decision Comparison: Baseline vs Explain-Aware")
plt.ylabel("Attack Detection Rate")
plt.tight_layout()
plt.show()

# Missed attack reduction
comparison = safe_results.copy()
comparison["Missed_Attack_Reduction"] = (
    comparison["Baseline_missed_attacks"] - comparison["ExplainAware_missed_attacks"]
)

plt.figure(figsize=(8, 5))
sns.barplot(data=comparison, x="Dataset", y="Missed_Attack_Reduction")
plt.title("Reduction in Missed Attacks Using Explanations")
plt.tight_layout()
plt.show()

# False block rate comparison
fbr_plot = safe_results.melt(
    id_vars="Dataset",
    value_vars=["Baseline_false_block_rate", "ExplainAware_false_block_rate"],
    var_name="Method", value_name="FalseBlockRate"
)
fbr_plot["Method"] = fbr_plot["Method"].map({
    "Baseline_false_block_rate": "Baseline",
    "ExplainAware_false_block_rate": "Explain-Aware"
})

plt.figure(figsize=(10, 6))
sns.barplot(data=fbr_plot, x="Dataset", y="FalseBlockRate", hue="Method")
plt.title("False Block Rate Comparison")
plt.tight_layout()
plt.show()

## 17. Multi-Seed Robustness Evaluation

In [ ]:
# Run safe decision evaluation with different random seeds
# to compute confidence intervals

print(f"Running multi-seed evaluation with seeds: {RANDOM_SEEDS}\n")

multiseed_results = []

for seed in RANDOM_SEEDS:
    for ds_name, (X_ds, y_ds) in [
        ("Host-Test", (X_host_test, y_host_test)),
        ("Time-Test", (X_time_test, y_time_test)),
        ("Hard-Test", (X_hard_test, y_hard_test)),
    ]:
        X_s, y_s = sample_df(X_ds, y_ds, MAX_SHAP_ROWS, seed)

        proba = lgb_calibrated.predict_proba(X_s)
        pred_idx = np.argmax(proba, axis=1)

        shap_raw = explainer.shap_values(X_s)
        shap_vecs = convert_shap_to_predicted_class_vectors(shap_raw, pred_idx)
        pred_sim = compute_pred_similarities_vectorized(shap_vecs, pred_idx, train_sig_by_class_idx)

        safe = evaluate_safe_decision_policy(
            proba=proba, pred_idx=pred_idx, true_idx=y_s,
            benign_idx=benign_idx, pred_signature_similarity=pred_sim,
            prob_threshold=prob_threshold, sim_threshold=sim_threshold
        )

        multiseed_results.append({
            "Seed": seed,
            "Dataset": ds_name,
            "attack_not_missed_rate": safe["attack_not_missed_rate"],
            "false_block_rate": safe["false_block_rate"],
            "escalation_rate": safe["escalation_rate"],
            "missed_attacks": safe["missed_attacks"],
        })

multiseed_df = pd.DataFrame(multiseed_results)

# Summary with mean +/- std
summary = multiseed_df.groupby("Dataset").agg(
    attack_detection_mean=("attack_not_missed_rate", "mean"),
    attack_detection_std=("attack_not_missed_rate", "std"),
    false_block_mean=("false_block_rate", "mean"),
    false_block_std=("false_block_rate", "std"),
    escalation_mean=("escalation_rate", "mean"),
    escalation_std=("escalation_rate", "std"),
).round(4)

print("Multi-seed results (mean +/- std):")
display(summary)
display(multiseed_df.round(4))

## 18. Save All Results & Deployment Artifact

In [ ]:
timing_df = pd.DataFrame(inference_times)
# Save tables
metrics_df.to_csv(os.path.join(DRIVE_RESULTS_PATH, "model_detection_metrics.csv"), index=False)
safe_results.to_csv(os.path.join(DRIVE_RESULTS_PATH, "safe_decision_results.csv"), index=False)
timing_df.to_csv(os.path.join(DRIVE_RESULTS_PATH, "inference_timing.csv"), index=False)
multiseed_df.to_csv(os.path.join(DRIVE_RESULTS_PATH, "multiseed_results.csv"), index=False)
threshold_search_df.to_csv(os.path.join(DRIVE_RESULTS_PATH, "threshold_search.csv"), index=False)

# Save models
joblib.dump(lgb_model, os.path.join(DRIVE_RESULTS_PATH, "lightgbm_model.pkl"))
joblib.dump(xgb_model, os.path.join(DRIVE_RESULTS_PATH, "xgboost_model.pkl"))
joblib.dump(cb_model, os.path.join(DRIVE_RESULTS_PATH, "catboost_model.pkl"))
joblib.dump(rf_model, os.path.join(DRIVE_RESULTS_PATH, "randomforest_model.pkl"))
joblib.dump(mlp_model, os.path.join(DRIVE_RESULTS_PATH, "mlp_model.pkl"))


# *** Unified Deployment Artifact ***
# Everything needed for inference in a single file

deployment_artifact = {
    "lgb_model": lgb_model,
    "lgb_calibrated": lgb_calibrated,
    "explainer_model": lgb_model,  # TreeExplainer will be recreated
    "label_encoder": le,
    "selected_features": selected_features,
    "train_sig_by_class_idx": train_sig_by_class_idx,
    "prob_threshold": float(prob_threshold),
    "sim_threshold": float(sim_threshold),
    "benign_idx": int(benign_idx),
    "rf_imputer": rf_imputer,
}
joblib.dump(deployment_artifact, os.path.join(DRIVE_RESULTS_PATH, "deployment_artifact.pkl"))


# Save config
config = {
    "dataset": "BCCC-CIC-IDS-2017",
    "models": ["LightGBM", "XGBoost", "CatBoost", "RandomForest", "MLP"],
    "random_seeds": RANDOM_SEEDS,
    "feature_selection_top_n": FEATURE_SELECTION_TOP_N,
    "selected_features": selected_features,
    "top_k_signature_features": TOP_K_SIGNATURE_FEATURES,
    "min_rows_per_class_for_signature": MIN_ROWS_PER_CLASS_FOR_SIGNATURE,
    "max_shap_rows": MAX_SHAP_ROWS,
    "min_class_count": MIN_CLASS_COUNT,
    "seen_host_ratio": SEEN_HOST_RATIO,
    "prob_threshold": float(prob_threshold),
    "sim_threshold": float(sim_threshold),
    "min_attack_detection_rate": MIN_ATTACK_DETECTION_RATE,
    "calibration": "isotonic",
    "xai_methods": ["SHAP (TreeExplainer)", "LIME (cross-validation)"],
}
with open(os.path.join(DRIVE_RESULTS_PATH, "experiment_config.json"), "w") as f:
    json.dump(config, f, indent=4)

print("\nSaved files:")
for f in sorted(os.listdir(DRIVE_RESULTS_PATH)):
    size = os.path.getsize(os.path.join(DRIVE_RESULTS_PATH, f))
    print(f"  {f:45s} {size/1024:.1f} KB")